# 06 - Evaluation

## Stage 7 Objective

Evaluate simplifier and clause classifier outputs, generate report-ready charts, save aggregate metrics, and create a human evaluation CSV template.

## Inputs

- `outputs/predictions/simplifier_predictions.csv`
- `outputs/predictions/clause_classifier_predictions.csv`

## Outputs

- `data/evaluation/results.csv`
- `data/evaluation/human_eval_template.csv`
- charts and confusion matrix files in `outputs/charts/`

## 1. Configure Paths and Imports

In [ ]:
from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluator import (
    build_human_eval_template,
    compute_bertscore,
    compute_classifier_metrics,
    compute_rouge_scores,
    confusion_matrix_records,
    get_human_eval_columns,
    get_result_columns,
    summarize_simplification_quality,
    write_csv,
)

SIMPLIFIER_PREDICTIONS_PATH = PROJECT_ROOT / 'outputs' / 'predictions' / 'simplifier_predictions.csv'
CLASSIFIER_PREDICTIONS_PATH = PROJECT_ROOT / 'outputs' / 'predictions' / 'clause_classifier_predictions.csv'
SIMPLIFICATION_DATASET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'simplification_dataset.csv'
CLASSIFICATION_DATASET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'classification_dataset.csv'
RESULTS_PATH = PROJECT_ROOT / 'data' / 'evaluation' / 'results.csv'
HUMAN_EVAL_PATH = PROJECT_ROOT / 'data' / 'evaluation' / 'human_eval_template.csv'
CHARTS_DIR = PROJECT_ROOT / 'outputs' / 'charts'
CONFUSION_MATRIX_CSV_PATH = CHARTS_DIR / 'clause_classifier_confusion_matrix.csv'
CONFUSION_MATRIX_PNG_PATH = CHARTS_DIR / 'clause_classifier_confusion_matrix.png'

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
HUMAN_EVAL_PATH.parent.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Results path: {RESULTS_PATH}')
print(f'Charts directory: {CHARTS_DIR}')

## 2. Load Prediction Files

In [ ]:
def read_csv_or_empty(path, columns):
    if path.exists():
        df = pd.read_csv(path)
    else:
        df = pd.DataFrame(columns=columns)
    for column in columns:
        if column not in df.columns:
            df[column] = ''
    return df[columns]

simplifier_columns = ['clause_id', 'split', 'clause_text', 'reference_simple_clause', 'predicted_simple_clause']
classifier_columns = ['clause_id', 'split', 'clause_text', 'true_clause_type', 'predicted_clause_type', 'confidence']

simplifier_df = read_csv_or_empty(SIMPLIFIER_PREDICTIONS_PATH, simplifier_columns)
classifier_df = read_csv_or_empty(CLASSIFIER_PREDICTIONS_PATH, classifier_columns)

for column in simplifier_columns:
    simplifier_df[column] = simplifier_df[column].fillna('').astype(str)
for column in classifier_columns:
    classifier_df[column] = classifier_df[column].fillna('').astype(str)

print(f'Simplifier predictions: {len(simplifier_df)} row(s)')
print(f'Classifier predictions: {len(classifier_df)} row(s)')
display(simplifier_df.head())
display(classifier_df.head())

## 3. Evaluate Simplifier Metrics

In [ ]:
results = []

simplifier_eval_df = simplifier_df[
    (simplifier_df['reference_simple_clause'].str.strip() != '')
    & (simplifier_df['predicted_simple_clause'].str.strip() != '')
].copy()

rouge_scores, rouge_note = compute_rouge_scores(
    simplifier_eval_df['predicted_simple_clause'].tolist(),
    simplifier_eval_df['reference_simple_clause'].tolist(),
)
for metric, value in rouge_scores.items():
    results.append({'stage': 'simplifier', 'metric': metric, 'value': value, 'notes': rouge_note})

bertscore_values, bertscore_note = compute_bertscore(
    simplifier_eval_df['predicted_simple_clause'].tolist(),
    simplifier_eval_df['reference_simple_clause'].tolist(),
)
for metric, value in bertscore_values.items():
    results.append({'stage': 'simplifier', 'metric': metric, 'value': value, 'notes': bertscore_note})

readability_summary = summarize_simplification_quality(simplifier_eval_df.to_dict(orient='records'))
for metric, value in readability_summary.items():
    results.append({'stage': 'simplifier', 'metric': metric, 'value': value, 'notes': 'readability/compression summary'})

if simplifier_eval_df.empty:
    print('No complete simplifier prediction/reference pairs found. Run Notebook 04 to populate simplifier predictions.')
else:
    display(pd.DataFrame([rouge_scores | bertscore_values | readability_summary]))

## 4. Simplifier Charts

In [ ]:
def is_number(value):
    try:
        return not math.isnan(float(value))
    except Exception:
        return False

def save_no_data_chart(path, title):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.text(0.5, 0.5, 'No prediction data available', ha='center', va='center')
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

if not simplifier_eval_df.empty:
    readability_chart_path = CHARTS_DIR / 'simplifier_readability_before_after.png'
    compression_chart_path = CHARTS_DIR / 'simplifier_compression_ratio.png'
    metric_chart_path = CHARTS_DIR / 'simplifier_rouge_bertscore.png'

    readability_values = [
        readability_summary['source_flesch_kincaid_grade'],
        readability_summary['prediction_flesch_kincaid_grade'],
    ]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(['Before', 'After'], readability_values, color=['#4C78A8', '#59A14F'])
    ax.set_ylabel('Flesch-Kincaid grade')
    ax.set_title('Simplifier Readability Before and After')
    fig.tight_layout()
    fig.savefig(readability_chart_path, dpi=150)
    plt.show()

    compression_values = [
        len(predicted) / max(len(original), 1)
        for original, predicted in zip(simplifier_eval_df['clause_text'], simplifier_eval_df['predicted_simple_clause'])
    ]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(compression_values, bins=min(10, max(1, len(compression_values))), color='#F28E2B', edgecolor='black')
    ax.set_xlabel('Compression ratio')
    ax.set_ylabel('Clause count')
    ax.set_title('Simplifier Compression Ratio')
    fig.tight_layout()
    fig.savefig(compression_chart_path, dpi=150)
    plt.show()

    metric_values = {**rouge_scores, **bertscore_values}
    metric_values = {key: value for key, value in metric_values.items() if is_number(value)}
    if metric_values:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(metric_values.keys(), metric_values.values(), color='#76B7B2')
        ax.set_ylim(0, 1)
        ax.set_ylabel('Score')
        ax.set_title('Simplifier ROUGE and BERTScore')
        ax.tick_params(axis='x', rotation=30)
        fig.tight_layout()
        fig.savefig(metric_chart_path, dpi=150)
        plt.show()

    print(f'Saved simplifier charts to {CHARTS_DIR.relative_to(PROJECT_ROOT)}')
else:
    save_no_data_chart(CHARTS_DIR / 'simplifier_readability_before_after.png', 'Simplifier Readability Before and After')
    save_no_data_chart(CHARTS_DIR / 'simplifier_compression_ratio.png', 'Simplifier Compression Ratio')
    save_no_data_chart(CHARTS_DIR / 'simplifier_rouge_bertscore.png', 'Simplifier ROUGE and BERTScore')
    print('Saved placeholder simplifier charts because no complete simplifier predictions are available.')

## 5. Evaluate Classifier Metrics

In [ ]:
classifier_eval_df = classifier_df[
    (classifier_df['true_clause_type'].str.strip() != '')
    & (classifier_df['predicted_clause_type'].str.strip() != '')
].copy()

classifier_metrics = compute_classifier_metrics(
    classifier_eval_df['true_clause_type'].tolist(),
    classifier_eval_df['predicted_clause_type'].tolist(),
)
for metric, value in classifier_metrics.items():
    results.append({'stage': 'classifier', 'metric': metric, 'value': value, 'notes': 'weighted average where applicable'})

if classifier_eval_df.empty:
    print('No complete classifier true/predicted label pairs found. Run Notebook 05 to populate classifier predictions.')
else:
    display(pd.DataFrame([classifier_metrics]))

## 6. Confusion Matrix and Classifier Charts

In [ ]:
if not classifier_eval_df.empty:
    labels, matrix = confusion_matrix_records(
        classifier_eval_df['true_clause_type'].tolist(),
        classifier_eval_df['predicted_clause_type'].tolist(),
    )
    confusion_df = pd.DataFrame(matrix, index=[f'true_{label}' for label in labels], columns=[f'pred_{label}' for label in labels])
    confusion_df.to_csv(CONFUSION_MATRIX_CSV_PATH)
    display(confusion_df)

    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.2), max(4, len(labels) * 0.9)))
    image = ax.imshow(confusion_df.values, cmap='Blues')
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted clause_type')
    ax.set_ylabel('True clause_type')
    ax.set_title('Clause Classifier Confusion Matrix')
    for row_index in range(confusion_df.shape[0]):
        for col_index in range(confusion_df.shape[1]):
            ax.text(col_index, row_index, confusion_df.iloc[row_index, col_index], ha='center', va='center')
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(CONFUSION_MATRIX_PNG_PATH, dpi=150)
    plt.show()

    classifier_chart_path = CHARTS_DIR / 'clause_classifier_metrics.png'
    clean_metrics = {key: value for key, value in classifier_metrics.items() if is_number(value)}
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(clean_metrics.keys(), clean_metrics.values(), color='#E15759')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score')
    ax.set_title('Clause Classifier Metrics')
    fig.tight_layout()
    fig.savefig(classifier_chart_path, dpi=150)
    plt.show()

    print(f'Saved classifier charts and confusion matrix to {CHARTS_DIR.relative_to(PROJECT_ROOT)}')
else:
    empty_confusion = pd.DataFrame()
    empty_confusion.to_csv(CONFUSION_MATRIX_CSV_PATH)
    save_no_data_chart(CONFUSION_MATRIX_PNG_PATH, 'Clause Classifier Confusion Matrix')
    save_no_data_chart(CHARTS_DIR / 'clause_classifier_metrics.png', 'Clause Classifier Metrics')
    print('Saved placeholder classifier charts because no complete classifier predictions are available.')

## 7. Save Evaluation Results

In [ ]:
results_df = pd.DataFrame(results, columns=get_result_columns())
results_df.to_csv(RESULTS_PATH, index=False)
print(f'Saved {len(results_df)} result row(s) to {RESULTS_PATH.relative_to(PROJECT_ROOT)}')
display(results_df)

## 8. Build Human Evaluation Template

In [ ]:
def fallback_simplifier_rows():
    if not SIMPLIFICATION_DATASET_PATH.exists():
        return []
    df = pd.read_csv(SIMPLIFICATION_DATASET_PATH).fillna('')
    return pd.DataFrame(
        {
            'clause_id': df.get('clause_id', ''),
            'split': df.get('split', ''),
            'clause_text': df.get('clause_text', ''),
            'reference_simple_clause': df.get('simple_clause', ''),
            'predicted_simple_clause': '',
        }
    ).to_dict(orient='records')

def fallback_classifier_rows():
    if not CLASSIFICATION_DATASET_PATH.exists():
        return []
    df = pd.read_csv(CLASSIFICATION_DATASET_PATH).fillna('')
    return pd.DataFrame(
        {
            'clause_id': df.get('clause_id', ''),
            'split': df.get('split', ''),
            'clause_text': df.get('clause_text', ''),
            'true_clause_type': df.get('clause_type', ''),
            'predicted_clause_type': '',
        }
    ).to_dict(orient='records')

human_simplifier_rows = simplifier_df.to_dict(orient='records') if not simplifier_df.empty else fallback_simplifier_rows()
human_classifier_rows = classifier_df.to_dict(orient='records') if not classifier_df.empty else fallback_classifier_rows()

human_eval_rows = build_human_eval_template(
    human_simplifier_rows,
    human_classifier_rows,
    sample_size=20,
)
if human_eval_rows:
    base_rows = list(human_eval_rows)
    while len(human_eval_rows) < 20:
        source = dict(base_rows[len(human_eval_rows) % len(base_rows)])
        source['evaluation_id'] = f'eval_{len(human_eval_rows) + 1:03d}'
        source['notes'] = 'duplicate review slot from small sample data'
        human_eval_rows.append(source)
human_eval_rows = human_eval_rows[:20]
write_csv(HUMAN_EVAL_PATH, human_eval_rows, get_human_eval_columns())

human_eval_df = pd.DataFrame(human_eval_rows, columns=get_human_eval_columns())
print(f'Saved {len(human_eval_df)} human evaluation row(s) to {HUMAN_EVAL_PATH.relative_to(PROJECT_ROOT)}')
display(human_eval_df.head(20))